# GCSI Headline Feed Live Polling

## Integration guide: Global Commodity Sentiment Indices, headline feed

The headline feed is the input to the hourly index: one row per headline × commodity × topic, each with a
**match type** and **three horizon scores**. Use it when you want your own aggregation windows, your own
filters, or the headlines behind a move. The same pattern, as three Docker services, is in
`app/live_headline_polling/`.

### Output schema

| Field | Description |
|---|---|
| `headline`, `link`, `source_name` | The article (publisher content: use it under your licence, do not republish) |
| `ticker` | The commodity |
| `match_type` | `EXPLICIT`, `RELATED_ENTITY` or `MACROECONOMIC` |
| `topic` | Topic label, `Sector::L1::L2` |
| `topic_probability` | Classifier confidence (0–1) that the headline belongs to the topic. This is not sentiment. |
| `short_term_sentiment_score` / `medium_term_…` / `long_term_…` | Horizon scores from −1 to +1, positive = bullish for price |
| `extraction_time` | When Permutable received it (UTC). The hourly index is binned on this field. |

### What this notebook shows

1. Polling the feed into SQLite without duplicates
2. The match-type mix, and why `topic_probability` is a filter rather than a score
3. How the three horizons relate to each other
4. Building your own 15-minute index, binned on **receipt time** so it matches the published hourly index

> **Disclaimer:** This notebook is provided for informational and research purposes only. Nothing here constitutes financial advice or a recommendation to buy, sell, or hold any asset. Sentiment scores are model outputs describing how news was judged at publication; they are not price forecasts.

In [ ]:
%pip install requests pandas numpy matplotlib seaborn

## Step 1: Configuration

In [ ]:
import getpass
import hashlib
import os
import random
import sqlite3
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns


def load_env_file() -> None:
    """Read KEY=VALUE pairs from the nearest .env file without overriding variables already set."""
    for base in [Path.cwd(), *Path.cwd().parents]:
        env_path = base / '.env'
        if env_path.exists():
            for line in env_path.read_text().splitlines():
                line = line.strip()
                if line and not line.startswith('#') and '=' in line:
                    key, value = line.split('=', 1)
                    os.environ.setdefault(key.strip(), value.strip().strip('"'))
            return


def get_api_key() -> str:
    load_env_file()
    key = os.environ.get('API_KEY') or os.environ.get('PERMUTABLE_API_KEY')
    if not key:
        key = getpass.getpass('Enter your Permutable AI API key: ')
    return key


API_KEY = get_api_key()

TICKERS = ['BZ_COM']
MATCH_TYPES = [None]               # None uses the API default COMBINED; or e.g. ['EXPLICIT', 'RELATED_ENTITY']
BACKFILL_HOURS = 48
POLL_INTERVAL_SECONDS = 900
MIN_TOPIC_PROBABILITY = 0.0        # raise to keep only confident topic assignments
AGG_FREQ = '15min'
DB_PATH = Path('gcsi_headlines_nb.db')

HORIZONS = ['short_term_sentiment_score', 'medium_term_sentiment_score', 'long_term_sentiment_score']

BASE_URL = os.environ.get('BASE_URL', 'https://copilot-api.permutable.ai/v1')

# GCSI routes. The ticker and topic are query presets, not part of the path.
FEED_LIVE_PATH = '/headlines/feed/live/gcsi'
FEED_HISTORICAL_PATH = '/headlines/feed/historical/gcsi'
PAGE_LIMIT = 1000
MAX_LOOKBACK_DAYS = 90

## Step 2: Fetch and storage helpers

The feed has no row id, so each row gets a key built from (ticker, topic, match type, link or headline,
publication time). Polling the same window twice is then harmless.

In [ ]:
SESSION = requests.Session()
SESSION.headers.update({'x-api-key': API_KEY, 'accept': 'application/json'})


def api_get(path: str, params: dict | None = None, max_retries: int = 5):
    """GET a Permutable API path, backing off exponentially (with jitter) on 429 and 5xx responses."""
    params = {k: v for k, v in (params or {}).items() if v is not None}
    delay = 1.0
    for attempt in range(max_retries + 1):
        response = SESSION.get(BASE_URL + path, params=params, timeout=60)
        if response.status_code == 429 or response.status_code >= 500:
            if attempt == max_retries:
                response.raise_for_status()
            time.sleep(delay + random.uniform(0, delay / 2))
            delay *= 2
            continue
        if response.status_code in (401, 403):
            raise PermissionError(
                f'{response.status_code} on {path}: check the API key, and that your licence includes GCSI.'
            )
        response.raise_for_status()
        return response.json()


def _records(body) -> list[dict]:
    """Live endpoints return a list; historical endpoints return {data, has_more, next_token}."""
    return body['data'] if isinstance(body, dict) else body


def _stamp(records: list[dict], **dims) -> list[dict]:
    """Responses can omit the dimensions you asked for, so write them back onto each record."""
    for record in records:
        for key, value in dims.items():
            if value is not None:
                record.setdefault(key, value)
    return records


def _paginate(path: str, params: dict) -> list[dict]:
    params = {**params, 'limit': PAGE_LIMIT}
    rows: list[dict] = []
    while True:
        body = api_get(path, params)
        rows.extend(_records(body))
        if not (isinstance(body, dict) and body.get('has_more')):
            return rows
        params['next_token'] = body['next_token']


TOPIC_SEPARATOR = '::'  # Sector::L1::L2, the form the API returns


def normalise_topic(labels: pd.Series) -> pd.Series:
    """Write each label as sector, driver and topic joined by TOPIC_SEPARATOR, whatever separator the source used."""
    return labels.astype('string').str.replace(r'\s*(?:::|-)\s*', TOPIC_SEPARATOR, regex=True)


def fetch_live_feed(ticker: str, match_type: str | None = None) -> list[dict]:
    params = {'ticker_preset': ticker, 'match_type': match_type, 'limit': PAGE_LIMIT}
    return _stamp(_records(api_get(FEED_LIVE_PATH, params)), ticker=ticker)


def fetch_historical_feed(ticker: str, start: datetime, end: datetime | None = None, match_type: str | None = None) -> list[dict]:
    params = {'ticker_preset': ticker, 'match_type': match_type, 'start_date': start.date().isoformat(),
              'end_date': end.date().isoformat() if end else None, 'limit': PAGE_LIMIT}
    return _stamp(_paginate(FEED_HISTORICAL_PATH, params), ticker=ticker)


FEED_COLUMNS = ['row_key', 'ticker', 'match_type', 'topic', 'topic_probability', *HORIZONS,
                'headline', 'link', 'source_name', 'extraction_time', 'fetched_at']


def row_key(record: dict) -> str:
    parts = [record.get(k) or '' for k in ('ticker', 'topic', 'match_type', 'extraction_time')]
    parts.append(record.get('link') or record.get('headline') or '')
    return hashlib.sha1('|'.join(map(str, parts)).encode()).hexdigest()


def setup_database(db_path: Path = DB_PATH) -> None:
    columns = ', '.join(f'{c} TEXT' if c not in ('topic_probability', *HORIZONS) else f'{c} REAL' for c in FEED_COLUMNS)
    with sqlite3.connect(db_path) as conn:
        conn.execute(f'CREATE TABLE IF NOT EXISTS gcsi_headlines ({columns}, PRIMARY KEY (row_key))')


def upsert_feed(records: list[dict], db_path: Path = DB_PATH) -> int:
    if not records:
        return 0
    fetched_at = datetime.now(timezone.utc).isoformat()
    rows = [tuple(row_key(r) if c == 'row_key' else fetched_at if c == 'fetched_at' else r.get(c) for c in FEED_COLUMNS)
            for r in records]
    with sqlite3.connect(db_path) as conn:
        conn.executemany(
            f'INSERT OR REPLACE INTO gcsi_headlines ({", ".join(FEED_COLUMNS)}) VALUES ({", ".join("?" * len(FEED_COLUMNS))})',
            rows,
        )
    return len(rows)


def load_feed(hours: int = BACKFILL_HOURS) -> pd.DataFrame:
    with sqlite3.connect(DB_PATH) as conn:
        frame = pd.read_sql('SELECT * FROM gcsi_headlines', conn)
    frame['extraction_time'] = pd.to_datetime(frame['extraction_time'], utc=True, format='ISO8601')
    frame['topic'] = normalise_topic(frame['topic'])
    since = pd.Timestamp.now(tz='UTC') - pd.Timedelta(hours=hours)
    return frame[frame['extraction_time'] >= since].reset_index(drop=True)


setup_database()
print('Database ready:', DB_PATH.resolve())

## Step 3: Dry run and backfill

In [ ]:
sample = fetch_live_feed(TICKERS[0])
print(f'{len(sample)} live rows for {TICKERS[0]}; fields: {sorted(sample[0]) if sample else "none"}')

start = datetime.now(timezone.utc) - timedelta(hours=BACKFILL_HOURS)
for ticker in TICKERS:
    for match_type in MATCH_TYPES:
        n = upsert_feed(fetch_historical_feed(ticker, start, match_type=match_type))
        print(f'{ticker} / {match_type or "all match types"}: {n:,} rows upserted')

feed = load_feed()
feed = feed[feed['topic_probability'] >= MIN_TOPIC_PROBABILITY]
print(f'{len(feed):,} rows | {feed["link"].nunique():,} distinct links | {feed["topic"].nunique()} topics')

## Step 4: Match types and topic probability

One headline can appear several times for a ticker: once per topic. Within a ticker, each headline has a single
match type (priority `EXPLICIT`, then `RELATED_ENTITY`, then `MACROECONOMIC`).
`topic_probability` measures how sure the classifier is about the *topic*. A low value does not mean low
sentiment, so use it to trade coverage for precision.

In [ ]:
mix = feed.groupby('match_type').agg(rows=('row_key', 'size'), links=('link', 'nunique'),
                                      avg_short_term=('short_term_sentiment_score', 'mean')).round(3)
display(mix)

thresholds = [0.0, 0.5, 0.7, 0.9]
kept = pd.DataFrame({
    'min_topic_probability': thresholds,
    'rows_kept': [(feed['topic_probability'] >= t).sum() for t in thresholds],
    'avg_short_term': [feed.loc[feed['topic_probability'] >= t, 'short_term_sentiment_score'].mean() for t in thresholds],
}).round(3)
display(kept)

## Step 5: The three horizons

Each horizon judges a different part of the futures curve, so they need not agree. The scatter shows how
short-term and long-term scores relate for the same headlines.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for horizon in HORIZONS:
    axes[0].hist(feed[horizon].dropna(), bins=60, alpha=0.5, label=horizon.replace('_sentiment_score', ''))
axes[0].set_title('Score distribution by horizon')
axes[0].legend()
axes[1].scatter(feed['short_term_sentiment_score'], feed['long_term_sentiment_score'], s=6, alpha=0.3)
axes[1].axhline(0, color='grey', lw=0.6)
axes[1].axvline(0, color='grey', lw=0.6)
axes[1].set_xlabel('short_term')
axes[1].set_ylabel('long_term')
axes[1].set_title('Short vs long horizon, same headline')
plt.tight_layout()
plt.show()
display(feed[HORIZONS].corr().round(2))

## Step 6: Your own index on receipt time

Bin on `extraction_time` (when Permutable received the headline), labelling each bin by its **end**, as the
hourly index does. The feed does not carry a publisher publication time.

One article can appear in several topics, so the `headline_count` here counts topic-links.

In [ ]:
def own_index(frame: pd.DataFrame, horizon: str = 'short_term_sentiment_score', freq: str = AGG_FREQ) -> pd.DataFrame:
    binned = frame.assign(bin_end=frame['extraction_time'].dt.floor(freq) + pd.Timedelta(freq))
    out = binned.groupby(['ticker', 'match_type', 'bin_end']).agg(
        headline_count=(horizon, 'size'),
        sentiment_sum=(horizon, 'sum'),
        sentiment_std=(horizon, 'std'),
    ).reset_index()
    out['sentiment_std'] = out['sentiment_std'].fillna(0.0)   # matches the index: 0 for a single headline
    out['sentiment_avg'] = out['sentiment_sum'] / out['headline_count']
    return out


own = own_index(feed)
fig, ax = plt.subplots(figsize=(14, 4))
for match_type, group in own.groupby('match_type'):
    rolled = group.set_index('bin_end')[['sentiment_sum', 'headline_count']].rolling('6h').sum()
    rolled = rolled[rolled.index >= rolled.index.min() + pd.Timedelta('6h')]   # drop the warm-up window
    ax.plot(rolled.index, rolled['sentiment_sum'] / rolled['headline_count'], label=match_type)
ax.axhline(0, color='grey', lw=0.8, ls=':')
ax.set_title(f'{TICKERS[0]} | short_term | 6h weighted average from {AGG_FREQ} bins, by match type')
ax.legend()
plt.tight_layout()
plt.show()

## Step 7: Topic heatmap

The busiest topics in the window, with cells under five topic-links left blank.

In [ ]:
top = feed['topic'].value_counts().head(12).index
sub = feed[feed['topic'].isin(top)].assign(hour=lambda f: f['extraction_time'].dt.floor('6h'))
grid = sub.groupby(['topic', 'hour'])['short_term_sentiment_score'].agg(['mean', 'size'])
heat = grid['mean'].where(grid['size'] >= 5).unstack()
heat.columns = heat.columns.strftime('%d %b\n%H:00')
fig, ax = plt.subplots(figsize=(14, 0.45 * len(heat) + 1.5))
sns.heatmap(heat, cmap='RdYlGn', center=0, linewidths=0.3, ax=ax, cbar_kws={'label': 'Average short-term score'})
ax.set_xlabel('6-hour receipt window (UTC)')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## Step 8: Strongest headlines

The headlines behind the most recent move, ranked by absolute short-term score. These are for your own analysis.
Publisher text should not be republished.

In [ ]:
recent = feed[feed['extraction_time'] >= feed['extraction_time'].max() - pd.Timedelta(hours=6)]
strongest = recent.reindex(recent['short_term_sentiment_score'].abs().sort_values(ascending=False).index)
display(strongest.drop_duplicates('link')[['extraction_time', 'match_type', 'topic', 'short_term_sentiment_score',
                                           'headline', 'source_name']].head(10))

## Step 9: Live polling loop

In [ ]:
RUN_POLLING_LOOP = False

if RUN_POLLING_LOOP:
    while True:
        started = datetime.now(timezone.utc)
        n = sum(upsert_feed(fetch_live_feed(t, m)) for t in TICKERS for m in MATCH_TYPES)
        print(f'{started:%H:%M:%S} UTC: {n} rows upserted')
        time.sleep(max(0, POLL_INTERVAL_SECONDS - (datetime.now(timezone.utc) - started).total_seconds()))
else:
    print('Polling loop disabled: set RUN_POLLING_LOOP = True to start it.')